# 10 — 2-D Occupancy Grid from Simulated Lidar

**Section:** Mapping · **Mirrors MATLAB:** *Occupancy Grid Utilities*

Build an occupancy grid by ray-casting simulated lidar from a few known robot poses. Each cell stores a **log-odds** value that is incremented along the ray (free) and at the hit cell (occupied):

$$\ell(c) \leftarrow \ell(c) + \log\frac{p(z\mid m_c)}{p(z\mid \neg m_c)}$$

Final probability: $p = 1 / (1 + e^{-\ell})$.

## Intuition — what's actually going on?

You're a robot with a lidar (laser scanner). At each pose, the lidar sweeps around and tells you the distance to whatever it hits in every direction. From multiple poses you want to build a **map** of the world.

The map is a grid of cells. Each cell holds a probability: how likely is it to be occupied? Every lidar beam that *passes through* a cell on its way to a hit is evidence that the cell is **free**. The cell that the beam *terminates on* is evidence that the cell is **occupied**.

The slick way to combine evidence from many beams is **log-odds**: instead of multiplying probabilities (slow, numerically unstable), we add log-likelihood-ratios (fast, numerically clean). The final probability is recovered via the logistic function at the end.

That's basically how every modern mobile robot builds a 2D map: ray-cast each scan, accumulate evidence in log-odds, view as probability.

### Compatibility check — math ↔ code

| Math | Code |
|---|---|
| Ray casting per beam $	heta + lpha$ | `for r in np.arange(step, max_range, step): x = pose[0]+r*cos(pose[2]+a); y = pose[1]+r*sin(pose[2]+a)` |
| Hit test against rectangle obstacles | `def hit_rect(x, y): for (x0,y0,x1,y1) in true_obs: if x0<=x<=x1 and y0<=y<=y1: return True` |
| Free-cell log-odds increment $\ell_	ext{free}$ | `log_odds[int(y/res), int(x/res)] += l_free` along ray |
| Occupied-cell log-odds increment $\ell_	ext{occ}$ | `log_odds[int(y/res), int(x/res)] += l_occ` at hit |
| Max-range "no hit" rays don't mark occupied | `if r < 12.0: ... log_odds[...] += l_occ` |
| Recover probability $p = \sigma(\ell)$ | `prob = 1 / (1 + np.exp(-log_odds))` |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

world = 20.0
true_obs = [(5, 5, 8, 7), (12, 12, 14, 16), (15, 4, 18, 7), (3, 14, 5, 17)]


def hit_rect(x, y):
    for (x0, y0, x1, y1) in true_obs:
        if x0 <= x <= x1 and y0 <= y <= y1:
            return True
    return False


def lidar(pose, n_beams=180, max_range=12.0, step=0.05):
    angles = np.linspace(-np.pi, np.pi, n_beams, endpoint=False)
    ranges = np.full(n_beams, max_range)
    for i, a in enumerate(angles):
        for r in np.arange(step, max_range, step):
            x = pose[0] + r * np.cos(pose[2] + a)
            y = pose[1] + r * np.sin(pose[2] + a)
            if hit_rect(x, y):
                ranges[i] = r
                break
    return angles, ranges


In [ ]:
res = 0.1
n = int(world / res)
log_odds = np.zeros((n, n))
l_occ, l_free = 0.7, -0.4

poses = [(3, 10, 0), (10, 3, np.pi / 2), (10, 17, -np.pi / 2), (17, 10, np.pi)]

for pose in poses:
    angles, ranges = lidar(pose, n_beams=120, max_range=12.0)
    for a, r in zip(angles, ranges):
        # Free cells along the ray
        for d in np.arange(0.1, r, res):
            x = pose[0] + d * np.cos(pose[2] + a)
            y = pose[1] + d * np.sin(pose[2] + a)
            if 0 <= x < world and 0 <= y < world:
                log_odds[int(y / res), int(x / res)] += l_free
        # Occupied cell at the hit (skip max-range "no hit" rays)
        if r < 12.0:
            x = pose[0] + r * np.cos(pose[2] + a)
            y = pose[1] + r * np.sin(pose[2] + a)
            if 0 <= x < world and 0 <= y < world:
                log_odds[int(y / res), int(x / res)] += l_occ

prob = 1 / (1 + np.exp(-log_odds))


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 6))

ax = axs[0]
for (x0, y0, x1, y1) in true_obs:
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, color='grey'))
for px, py, _ in poses:
    ax.plot(px, py, 'r*', markersize=16)
ax.set_xlim(0, world); ax.set_ylim(0, world)
ax.set_aspect('equal'); ax.grid(); ax.set_title('Ground truth + lidar poses')

axs[1].imshow(prob, origin='lower', cmap='Greys', extent=[0, world, 0, world])
for px, py, _ in poses:
    axs[1].plot(px, py, 'r*', markersize=16)
axs[1].set_title('Estimated occupancy (probability)')
axs[1].set_aspect('equal')
plt.tight_layout()
plt.show()


## References & rigor notes

**Bayes derivation of the log-odds update.** Let $\ell(c) = \log[p(m_c \mid z_{1:t}) / p(\neg m_c \mid z_{1:t})]$ be the log-odds. Under the (technically incorrect but workably accurate) **Moravec-Elfes approximation** that cells and beams are independent, Bayes' rule gives the recursive update
$$\ell_t(c) = \ell_{t-1}(c) + \log\frac{p(z_t \mid m_c)}{p(z_t \mid \neg m_c)},$$
which is exactly the additive form used here. The independence assumption is wrong (a ray casting through one cell continues through the next, coupling their posteriors), but the approximation makes the update tractable; the joint posterior over cells would otherwise be exponentially expensive. The log-odds form has two advantages: (a) no numerical underflow from multiplying tiny probabilities, (b) updates are pure addition.

**Inverse sensor model.** The constants $\ell_\text{occ}$ and $\ell_\text{free}$ derive from the sensor's miss probability $p_\text{miss}$ and false-alarm probability $p_\text{FA}$:
$$\ell_\text{occ} = \log\frac{1 - p_\text{miss}}{p_\text{FA}},\qquad \ell_\text{free} = \log\frac{p_\text{miss}}{1 - p_\text{FA}}.$$
Our chosen values $\ell_\text{occ} = 0.7$, $\ell_\text{free} = -0.4$ correspond to roughly $p_\text{miss} \approx 0.1$, $p_\text{FA} \approx 0.15$.

**Ray discretization.** Our step `0.05 m` is smaller than `res/√2 ≈ 0.07 m`, so every cell along a ray is visited; production code uses Bresenham's line algorithm for exact integer rasterization. Cells *past* max-range get no update (we don't know what's beyond).

**Complexity.** $O(R \cdot N_b)$ per scan, where $R$ = beam range / cell size and $N_b$ = beam count.

**References.**
- Moravec, H. P., & Elfes, A. (1985). *High resolution maps from wide angle sonar*. ICRA 1985.
- Thrun, S., Burgard, W., & Fox, D. (2005). *Probabilistic Robotics*, MIT Press, ch. 9 ("Occupancy Grid Mapping").
